In [ ]:
# 01 · STATE STAGE ACT 단일 정책 CONFIG · GitHub 중단 복구 · Drive 미사용
CFG = {
    # 저장 · ver2는 기존 DP 결과와 별도 Release에 저장
    'run_name': 'moveboxes_stage_policy_sweep_v1',
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes',
    'project_ref': '511c935a90738ab2a6de230cd928ae87dba7835d',
    'output_root': '/content/moveboxes_runs',

    # 데이터 / Colab 2026.07 · Python 3.12 · T4
    'repo_dir': '/content/berlin-marso-hackathon',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],

    # 학습 · 시뮬레이터 없이 CPU 데이터 + GPU 모델
    'seed': 42,
    'num_demos': None,
    'batch_size': 64,
    'lr': 0.0001,
    'total_iters': {'easy': 12000, 'medium': 20000, 'hard': 30000},
    'amp': True,

    # 작은 State ACT · 기존 DP 체크포인트 사용 불가
    'history': 16,
    'chunk_size': 16,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'latent_dim': 16,

    # 검증 / 중단 복구 / 작은 관측 위치 증강
    'save_freq': 1000,
    'warmup_steps': 500,
    'validation_batches': 8,
    'kl_weight': 0.001,
    'position_noise': 0.001,

    # 실행 · 매 스텝 재계획, 최근 XYZ 예측 평균, 집게는 최신 예측
    'temporal_decay': 0.25,
    'ensemble_window': 4,
    'ensemble_candidates': [1, 4],

    # 빠른 테스트 / 최종 평가 · 시드 분리, 기존 200스텝 유지
    'test_episodes': 8,
    'test_seed_start': 40000,
    'test_record_video': True,
    'tuning_episodes': 8,
    'tuning_seed_start': 20000,
    'benchmark_episodes': 100,
    'eval_seed_start': 30000,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},
    'record_eval_video': True,

    # 출력
    'console_interval_seconds': 10,
    'team': 'my-team',

    # 실행 조건으로 행동 학습 · 빠른 테스트가 0이면 긴 평가 생략
    'action_training_mode': 'prior',
    'repair_iters': 2000,
    'allow_zero_success_evaluation': False,

    # 단계 판단 · 학습된 완료/복구 확신이 낮으면 현재 단계 유지
    'gate_threshold': 0.65,
    'stage_threshold': 0.6,
    'stage_loss_weight': 0.3,
    'gate_loss_weight': 0.3,

    # 복구 시연 · 수집 전용 expert, 학습/제출은 학습된 정책
    'recovery_episodes': 16,
    'recovery_max_attempts': 48,
    'recovery_seed_start': 100000,
    'collection_max_steps': {'easy': 500, 'medium': 900, 'hard': 1400},
    'noise_probability': 0.08,
    'action_noise_std': 0.12,
    'drop_probability': 0.015,

}


In [ ]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','stage_chunk_policy',
             'stage_pick_sampling','stage_pick_train','stage_all_pick_retrain',
             'stage_pick_finetune','stage_pick_diagnose','stage_reference_check',
             'stage_anchor_continue','build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from stage_experiment import StageExperiment, source_bundle
experiment = StageExperiment(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


In [ ]:
# 03 · GitHub 인증 / 저장된 결과 복원
# 기존 환경변수 GH_TOKEN → Colab 보안 비밀 → 입력창 순서로 인증합니다.
# Fine-grained token: SongYunu/moveBoxes → Contents: Read and write.
# 이 저장소는 공개이므로 여기에 올린 모델·로그·영상도 공개됩니다.
import os
# 개인 사본에서 직접 지정할 경우 아래 한 줄의 주석을 풀어 사용하세요.
# os.environ['GH_TOKEN'] = '본인 토큰'
experiment.connect()
experiment.show_results()


In [ ]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


In [ ]:
# 05 · GitHub 데이터 다운로드·검증 / GPU와 정책 실행 확인
experiment.prepare_data()
experiment.check_runtime()


# Medium 34.4% 앵커 · 학습 없는 추론 설정 탐색

Medium 34.4%를 낸 `block_02.pt`의 SHA-256을 그대로 유지합니다. 과거 40.625%도 같은 가중치와 같은 기본 설정에서 다른 8개 시드로 측정된 값이므로 별도 우수 체크포인트가 아닙니다.

집기 직전 XY가 늦게 따라가는 현상에 직접 관련된 **temporal ensemble의 길이와 decay만** 6가지로 비교합니다. 먼저 8개 탐색 시드로 후보를 고른 뒤, 한 번도 선택에 쓰지 않은 16개 시드에서 기본 설정과 일대일 비교합니다. 새 설정이 최소 2개 상자를 더 분류해야 채택합니다. 가중치, stage/gate 임계값, Easy와 Hard는 바꾸지 않습니다.

각 실행은 공식 `eval.py`의 `SORT ACCURACY`만 읽습니다. 중간 결과는 GitHub Release에 백업을 시도하며, 같은 셀을 다시 실행하면 완료된 후보는 재사용합니다. Google Drive는 사용하지 않습니다.


In [ ]:
# 06 · 검증 앵커 복원 + 공식 eval.py 실행기
import hashlib, json, os, re, shutil, subprocess, sys
from pathlib import Path
from IPython.display import Video, display
from stage_anchor_continue import ANCHORS, package, prepare

MAX_STEPS = 200
SCREEN_SEEDS = list(range(64000, 64008))
CONFIRM_SEEDS = list(range(65000, 65016))
MIN_EXTRA_SORTED = 2
UPSTREAM = Path(CFG['repo_dir'])
OFFICIAL_DEFAULT = UPSTREAM/'conf/eval/default.yaml'
anchor_exp = prepare(experiment, run_suffix='_anchor_policy_sweep_base_v1')
RUN_DIR = Path(anchor_exp.run_dir)

BASELINE_OVERRIDES = dict(ensemble_window=4, temporal_decay=.25)
CANDIDATES = {
    'baseline_w4_d025': BASELINE_OVERRIDES,
    'responsive_w2_d025': dict(ensemble_window=2, temporal_decay=.25),
    'responsive_w2_d075': dict(ensemble_window=2, temporal_decay=.75),
    'responsive_w3_d075': dict(ensemble_window=3, temporal_decay=.75),
    'responsive_w4_d075': dict(ensemble_window=4, temporal_decay=.75),
    'responsive_w4_d150': dict(ensemble_window=4, temporal_decay=1.50),
}

def sha256(path):
    with Path(path).open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()

def make_candidate(name, overrides):
    kwargs = {} if overrides == BASELINE_OVERRIDES else {'policy_overrides':{'medium':overrides}}
    candidate = package(anchor_exp, folder_name='policy_sweep_candidates/'+name, **kwargs)
    manifest = json.loads((candidate/'manifest.json').read_text(encoding='utf-8'))
    assert manifest['levels']['medium']['checkpoint_sha256'] == ANCHORS['medium']['sha256']
    return candidate

def run_official(candidate, level, label, seeds=None, show_video=False):
    output = RUN_DIR/level/'policy_sweep_official_eval'/label
    output.mkdir(parents=True, exist_ok=True)
    if seeds is None:
        eval_config = OFFICIAL_DEFAULT
    else:
        eval_config = output/'eval_config.yaml'
        eval_config.write_text('eval:\n  n_episodes: '+str(len(seeds))+            '\n  seeds: '+json.dumps(seeds)+'\n', encoding='utf-8')
    checkpoint = candidate/'checkpoints'/level/'model.pt'
    sidecar = checkpoint.parent/'policy_config.json'
    identity = dict(level=level, label=label, seeds=seeds,
                    checkpoint_sha256=sha256(checkpoint), policy_sha256=sha256(sidecar))
    result_path = output/'result.json'
    if result_path.is_file():
        saved = json.loads(result_path.read_text(encoding='utf-8'))
        if saved.get('identity') == identity:
            print(f'[재사용] {label}: {saved["score"]:.3%}')
            return saved
    command = [sys.executable, str(UPSTREAM/'eval.py'), 'difficulty='+level,
        'obs_mode=state', 'policy=stage_policy:load_policy',
        'checkpoint='+str(checkpoint), 'eval_config='+str(eval_config),
        'max_episode_steps='+str(MAX_STEPS), 'hydra.run.dir='+str(output)]
    child_env = dict(os.environ)
    child_env['PYTHONPATH'] = str(candidate)+os.pathsep+str(UPSTREAM)+os.pathsep+child_env.get('PYTHONPATH','')
    for key in list(child_env):
        if key.startswith('MOVEBOXES_SYNC_'):
            child_env.pop(key, None)
    log = output/'official_eval.log'
    with log.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(command, cwd=UPSTREAM, env=child_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            errors='replace', bufsize=1)
        try:
            for line in process.stdout:
                handle.write(line); handle.flush(); print(line, end='', flush=True)
            code = process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                try: process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill(); process.wait()
            process.stdout.close()
    text = log.read_text(encoding='utf-8')
    match = re.search(r'SORT ACCURACY:\s+([0-9.]+)\s*%', text)
    if code or not match:
        raise RuntimeError(f'official eval failed ({code}); 로그: {log}')
    result = dict(identity=identity, score=float(match.group(1))/100, log=str(log),
                  candidate=str(candidate), policy=json.loads(sidecar.read_text(encoding='utf-8')))
    result_path.write_text(json.dumps(result, indent=2), encoding='utf-8')
    try:
        anchor_exp.sync_level(level)
    except Exception as error:
        print('GitHub 결과 백업 지연; 로컬 결과는 유지합니다:', error)
    if show_video:
        videos = sorted((output/'videos').rglob('*.mp4'), key=lambda p:p.stat().st_mtime)
        if videos:
            display(Video(str(videos[-1]), embed=True, width=900))
    return result


In [ ]:
# 07 · 8개 탐색 시드 · 6가지 반응성 설정 비교
SCREEN_RESULTS = {}
for name, overrides in CANDIDATES.items():
    candidate = make_candidate(name, overrides)
    result = run_official(candidate, 'medium', 'screen_'+name, SCREEN_SEEDS)
    result['overrides'] = overrides
    SCREEN_RESULTS[name] = result
    print(name, f'{result["score"]:.3%}', overrides)

# 동점이면 원래 설정을 우선하여 불필요한 변경을 막습니다.
SCREEN_WINNER = max(SCREEN_RESULTS,
    key=lambda name:(SCREEN_RESULTS[name]['score'], name == 'baseline_w4_d025'))
screen_summary = {name:row['score'] for name,row in SCREEN_RESULTS.items()}
(RUN_DIR/'medium/policy_sweep_screen.json').write_text(json.dumps(dict(
    seeds=SCREEN_SEEDS, scores=screen_summary, winner=SCREEN_WINNER), indent=2), encoding='utf-8')
print('탐색 승자:', SCREEN_WINNER, CANDIDATES[SCREEN_WINNER],
      f'{SCREEN_RESULTS[SCREEN_WINNER]["score"]:.3%}')


In [ ]:
# 08 · 새 16개 시드에서 baseline과 탐색 승자만 확인
BASELINE_CONFIRM = run_official(
    make_candidate('confirm_baseline', BASELINE_OVERRIDES),
    'medium', 'confirm_baseline', CONFIRM_SEEDS)

if SCREEN_WINNER == 'baseline_w4_d025':
    WINNER_CONFIRM = BASELINE_CONFIRM
    ACCEPT_TUNED_POLICY = False
else:
    winner_overrides = CANDIDATES[SCREEN_WINNER]
    WINNER_CONFIRM = run_official(
        make_candidate('confirm_'+SCREEN_WINNER, winner_overrides),
        'medium', 'confirm_'+SCREEN_WINNER, CONFIRM_SEEDS, show_video=True)
    parcel_trials = len(CONFIRM_SEEDS)*4
    extra_sorted = round((WINNER_CONFIRM['score']-BASELINE_CONFIRM['score'])*parcel_trials)
    ACCEPT_TUNED_POLICY = extra_sorted >= MIN_EXTRA_SORTED
    print('확인 시드 추가 정답 상자:', extra_sorted,
          '· 채택 기준:', MIN_EXTRA_SORTED, '· 채택:', ACCEPT_TUNED_POLICY)

SELECTED_OVERRIDES = CANDIDATES[SCREEN_WINNER] if ACCEPT_TUNED_POLICY else BASELINE_OVERRIDES
confirmation = dict(seeds=CONFIRM_SEEDS, screen_winner=SCREEN_WINNER,
    baseline_score=BASELINE_CONFIRM['score'], winner_score=WINNER_CONFIRM['score'],
    accepted=ACCEPT_TUNED_POLICY, selected_overrides=SELECTED_OVERRIDES)
(RUN_DIR/'medium/policy_sweep_confirmation.json').write_text(
    json.dumps(confirmation, indent=2), encoding='utf-8')
print(json.dumps(confirmation, indent=2))
try:
    anchor_exp.sync_level('medium')
except Exception as error:
    print('GitHub 결과 백업 지연; 로컬 결과는 유지합니다:', error)


In [ ]:
# 09 · 검증을 통과한 경우에만 설정 적용 · 전 난이도 기본 평가 · ZIP
policy_overrides = {'medium':SELECTED_OVERRIDES} if ACCEPT_TUNED_POLICY else {}
FINAL = package(anchor_exp, policy_overrides=policy_overrides,
                folder_name='final_policy_sweep_candidate')
manifest = json.loads((FINAL/'manifest.json').read_text(encoding='utf-8'))
assert manifest['levels']['medium']['checkpoint_sha256'] == ANCHORS['medium']['sha256']
expected_selection = 'policy_tuned_anchor' if ACCEPT_TUNED_POLICY else 'anchor'
assert manifest['levels']['medium']['selection'] == expected_selection

FINAL_RESULTS = {level:run_official(FINAL, level, 'final_default_'+level,
                                     show_video=(level == 'medium'))
                 for level in ('easy','medium','hard')}
print(json.dumps(dict(accepted=ACCEPT_TUNED_POLICY,
    medium_policy=manifest['levels']['medium']['policy_config'],
    final_default_scores={k:v['score'] for k,v in FINAL_RESULTS.items()}), indent=2))
archive = shutil.make_archive(str(RUN_DIR/'stage_act_validated_policy_submission'),
                              'zip', root_dir=FINAL)
from google.colab import files
print('제출 ZIP:', archive)
files.download(archive)
